# G-Eval 평가 실습 (OpenAI / Gemini 공통 환경)

- 이 노트북은 LLM을 **판사(judge)**로 사용해 모델의 답변을 자동 평가하는 G-Eval 패턴을 실습합니다.
- `ragas.ipynb`와 동일하게 **환경 변수 + OpenAI / Gemini 스위칭** 패턴을 그대로 사용합니다.
- 전체 흐름
  - 질문 / 기준 정답(Reference) / 모델 예측 답(Prediction) 준비
  - 평가용 LLM에게 여러 기준에 따른 점수와 이유를 요청
  - 결과를 테이블로 모아서 비교
- 필요한 환경 변수
  - OpenAI: `OPENAI_API_KEY`
  - Gemini: `GOOGLE_API_KEY` (Google AI Studio 키)
  - 선택: `EVALUATOR_PROVIDER` (지정 시 `"openai"` 또는 `"gemini"`)


In [1]:
# G-Eval에 사용할 기본 패키지 설치 (최초 1회 실행)
# - LangChain 본체 + OpenAI 연동
# - Gemini 연동용 langchain-google-genai (ragas 노트북과 동일 계열)
%pip install "langchain>=1.0.0" "langchain-openai>=1.0.0" "langchain-google-genai>=3.0.0" pandas python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 공통 환경 설정 및 평가용 LLM 선택 (OpenAI / Gemini 스위칭)
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# .env 또는 환경 변수 로드 (ragas 노트북과 동일 패턴)
load_dotenv()

# "openai" 또는 "gemini" 로 설정해서 스위칭
EVALUATOR_PROVIDER = "gemini"  # 또는 "openai"

if EVALUATOR_PROVIDER == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OpenAI를 사용하려면 OPENAI_API_KEY 환경 변수가 필요합니다.")

    JUDGE_MODEL_NAME = "gpt-4o-mini"

    judge_llm = ChatOpenAI(
        model=JUDGE_MODEL_NAME,
        temperature=0.0,
    )

elif EVALUATOR_PROVIDER == "gemini":
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("Gemini를 사용하려면 GOOGLE_API_KEY 환경 변수가 필요합니다.")

    # Gemini 패키지는 실제로 필요할 때만 import 합니다.
    from langchain_google_genai import ChatGoogleGenerativeAI

    JUDGE_MODEL_NAME = "gemini-2.5-flash"

    judge_llm = ChatGoogleGenerativeAI(
        model=JUDGE_MODEL_NAME,
        temperature=0.0,
    )

else:
    raise ValueError(f"지원하지 않는 EVALUATOR_PROVIDER 값: {EVALUATOR_PROVIDER}")

judge_llm

ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), temperature=0.0, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x1125b4f90>, default_metadata=(), model_kwargs={})

In [3]:
# G-Eval 평가 기준 및 LLM 프롬프트 정의 (내장 evaluator 모듈 사용하지 않고 직접 구현)
from typing import Any, Dict
import json

# 여러 기준을 한 번에 정의합니다.
GEVAL_CRITERIA: Dict[str, str] = {
    "correctness": "질문에 대해 사실적으로 올바른 답을 했는지 평가하세요.",
    "relevance": "질문과 직접적으로 관련된 내용만 포함했는지 평가하세요.",
    "clarity": "사용자가 이해하기 쉬운 명확한 표현인지 평가하세요.",
}


def build_geval_prompt(question: str, reference: str, prediction: str) -> str:
    """LLM이 일관된 JSON 스키마로 평가를 돌려주도록 프롬프트를 구성합니다."""
    criteria_text = "\n".join(
        f"- {name}: {description}" for name, description in GEVAL_CRITERIA.items()
    )
    return (
        "You are a strict evaluator for question-answer pairs.\n"
        "You must evaluate the model's prediction based on the given question and reference answer.\n\n"
        "Evaluation criteria:\n"
        f"{criteria_text}\n\n"
        "For each criterion, assign a score between 1 and 10 and explain the reason.\n"
        "Return ONLY a JSON object with the following structure and no extra text:\n"
        "{\n"
        '  "correctness": {"score": 1-10, "reason": "..."},\n'
        '  "relevance": {"score": 1-10, "reason": "..."},\n'
        '  "clarity": {"score": 1-10, "reason": "..."}\n'
        "}\n\n"
        f"QUESTION: {question}\n"
        f"REFERENCE_ANSWER: {reference}\n"
        f"PREDICTION: {prediction}\n"
    )


def strip_markdown_code_fence(text: str) -> str:
    """LLM이 ```json 코드 블록으로 감싼 응답에서 코드 펜스를 제거합니다."""
    stripped = text.strip()
    if not stripped.startswith("```"):
        return stripped

    lines = stripped.splitlines()
    # 첫 줄 (``` 또는 ```json ...) 제거
    if lines and lines[0].lstrip().startswith("```"):
        lines = lines[1:]
    # 마지막 줄에 ```가 있으면 제거
    if lines and lines[-1].strip().startswith("```"):
        lines = lines[:-1]
    return "\n".join(lines).strip()


def evaluate_answer_with_geval(
    llm: Any,
    *,
    question: str,
    reference: str,
    prediction: str,
) -> Dict[str, Any]:
    """하나의 Q/A 쌍에 대해 G-Eval 기준으로 점수/이유를 계산합니다."""
    prompt = build_geval_prompt(question=question, reference=reference, prediction=prediction)
    response = llm.invoke(prompt)

    content = getattr(response, "content", str(response))
    if not isinstance(content, str):
        content = str(content)

    # LLM이 ```json 코드 블록으로 감싼 경우를 먼저 정리
    content = strip_markdown_code_fence(content)

    try:
        return json.loads(content)
    except json.JSONDecodeError as exc:
        snippet = content[:200]
        raise ValueError(
            f"LLM 응답을 JSON으로 파싱하지 못했습니다. 내용 일부: {snippet!r}"
        ) from exc


# 간단 동작 확인용: 프롬프트 예시만 출력해 보고 싶을 때 사용
sample_prompt = build_geval_prompt(
    question="What is LangChain?",
    reference="LangChain is a framework for building applications with LLMs.",
    prediction="LangChain is a Python library for building LLM-powered applications.",
)
sample_prompt.split("\n")[:8]

['You are a strict evaluator for question-answer pairs.',
 "You must evaluate the model's prediction based on the given question and reference answer.",
 '',
 'Evaluation criteria:',
 '- correctness: 질문에 대해 사실적으로 올바른 답을 했는지 평가하세요.',
 '- relevance: 질문과 직접적으로 관련된 내용만 포함했는지 평가하세요.',
 '- clarity: 사용자가 이해하기 쉬운 명확한 표현인지 평가하세요.',
 '']

In [4]:
# 평가용 예제 데이터 (질문, 기준 정답, 모델 예측 답)
import pandas as pd

examples = [
    {
        "question": "What is LangChain?",
        "reference": "LangChain is a framework for building applications with LLMs.",
        "prediction": "LangChain is a Python library for building LLM-powered applications.",
    },
    {
        "question": "What is RAG?",
        "reference": "RAG is a technique that combines information retrieval with text generation.",
        "prediction": "RAG is retrieval-augmented generation that uses a retriever and a generator.",
    },
]

examples_df = pd.DataFrame(examples)
examples_df

,question,reference,prediction
0,What is LangChain?,LangChain is a framework for building applicat...,LangChain is a Python library for building LLM...
1,What is RAG?,RAG is a technique that combines information r...,RAG is retrieval-augmented generation that use...


In [5]:
# DataFrame 단위로 G-Eval 실행 및 기준별 점수/이유 정리
from typing import Any, Dict, List


def run_geval_on_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """여러 예제에 대해 G-Eval을 실행하고, 기준별 점수/이유를 테이블로 정리합니다."""
    rows: List[Dict[str, Any]] = []

    for _, row in df.iterrows():
        raw = evaluate_answer_with_geval(
            judge_llm,
            question=row["question"],
            reference=row["reference"],
            prediction=row["prediction"],
        )

        correctness = raw.get("correctness", {})
        relevance = raw.get("relevance", {})
        clarity = raw.get("clarity", {})

        rows.append(
            {
                "question": row["question"],
                "reference": row["reference"],
                "prediction": row["prediction"],
                "geval_raw": raw,
                "correctness_score": correctness.get("score"),
                "correctness_reason": correctness.get("reason"),
                "relevance_score": relevance.get("score"),
                "relevance_reason": relevance.get("reason"),
                "clarity_score": clarity.get("score"),
                "clarity_reason": clarity.get("reason"),
            }
        )

    return pd.DataFrame(rows)


geval_df = run_geval_on_dataframe(examples_df)

# 점수 위주로 먼저 확인
geval_df[[
    "question",
    "correctness_score",
    "relevance_score",
    "clarity_score",
]]

,question,correctness_score,relevance_score,clarity_score
0,What is LangChain?,10,10,10
1,What is RAG?,10,10,10


In [6]:
# 한 개 예제에 대한 G-Eval 상세 결과(JSON) 확인
from pprint import pprint

first_raw = geval_df.loc[0, "geval_raw"]
print("=== 첫 번째 예제 G-Eval 결과 ===")
pprint(first_raw)

=== 첫 번째 예제 G-Eval 결과 ===
{'clarity': {'reason': 'The prediction is clear, concise, and easy to '
                       'understand. It uses straightforward language to define '
                       'LangChain and its purpose.',
             'score': 10},
 'correctness': {'reason': 'The prediction accurately describes LangChain as a '
                           'Python library for building LLM-powered '
                           'applications. While the reference uses '
                           "'framework', 'Python library' is a correct and "
                           'more specific description of its primary '
                           'implementation, and it serves the same purpose as '
                           'described in the reference.',
                 'score': 10},
 'relevance': {'reason': "The prediction directly answers the question 'What "
                         "is LangChain?' by providing a concise and accurate "
                         'definition, without 